In [0]:
#restart python
dbutils.library.restartPython()

In [0]:
import sys
sys.path.append("/Workspace/Users/nalluriravi3@gmail.com/New_Medalion_Practice")
from utils.config_reader import ConfigReader
from utils.quarantine import QuarantineManager
from utils.datatype_update import DatatypeUpdate

## Read config file

In [0]:

config=ConfigReader.read_config("/Workspace/Users/nalluriravi3@gmail.com/New_Medalion_Practice/config/silver_config.json")
customer_config=config["customer"]

#create source and target dfs
source_df=spark.table(customer_config["source_table"])
target_df=spark.table(customer_config["target_table"])

# handle changes in data type
quarantine_df=spark.table(customer_config["quarantine_table"])

source_df=DatatypeUpdate.update_datatype(source_df,quarantine_df)

source_df.display()



## Create valid and invalid dfs and write invalid records into quarantine table

In [0]:
from pyspark.sql.functions import trim, initcap,col,upper,row_number,lit,current_timestamp,when
from pyspark.sql.window import Window

valid_df = source_df.withColumn(
    "customer_name", trim(upper(col(customer_config["mandatory_columns"][1])))
)

valid_df=valid_df.filter(
                (col("salary")>0) &
                (col("salary").isNotNull()) &
                (col(customer_config["mandatory_columns"][1]).isNotNull()) &
                (col(customer_config["mandatory_columns"][0]).isNotNull()) &
                (col("contact").isNotNull()) &
                (col("contact").rlike("^[0-9]{10}$")) &
                (col("email").isNotNull()) &
                (col("email").rlike("^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$")) 
)

# Remove duplicates from valid_df
valid_df = (
    valid_df
    .withColumn(
        "rn",
        row_number().over(
            Window.partitionBy(
                customer_config["mandatory_columns"][0]
            ).orderBy(
                col("ingestion_timestamp").desc()
            )
        )
    )
    .filter(col("rn") == 1)
    .drop("rn")
)

#create invalid df
invalid_df = source_df.filter(
    (col(customer_config["mandatory_columns"][1]).isNull()) |
    (col(customer_config["mandatory_columns"][0]).isNull()) |
    (col("email").isNull()) |
    (~col("email").rlike("^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$")) |
    (col("contact").isNull()) |
    (~col("contact").rlike("^[0-9]{10}$")) |
    (col("salary").isNull()) |
    (col("salary") <= 0)
)

# update rejected reason column
invalid_df = invalid_df.withColumn(
    "rejected_reason",
    when(
        col(customer_config["mandatory_columns"][0]).isNull(),
        "CUSTOMER_ID_NULL"
    )
    .when(
        col(customer_config["mandatory_columns"][1]).isNull(),
        "CUSTOMER_NAME_NULL"
    )
    .when(
        col("email").isNull(),
        "EMAIL_NULL"
    )
    .when(
        ~col("email").rlike(
            "^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\\.[A-Za-z]{2,}$"
        ),
        "INVALID_EMAIL"
    )
    .when(
        col("contact").isNull(),
        "CONTACT_NO_NULL"
    )
    .when(
        ~col("contact").rlike("^[0-9]{10}$"),
        "INVALID_CONTACT_NO"
    )
    .when(
        col("salary").isNull(),
        "SALARY_NULL"
    )
    .when(
        col("salary") <= 0,
        "INVALID_SALARY"
    )
).withColumn(
    "rejected_timestamp",
    current_timestamp()
)

#drop unwated columns
invalid_df = invalid_df.drop("ingestion_timestamp","postal_code","country","state","city")

#write into quarantine table
QuarantineManager.quarantine(invalid_df,customer_config["quarantine_table"])

display(valid_df)
display(invalid_df)


## Implement SCD2

In [0]:
from delta.tables import DeltaTable

changed_records_df =(
    valid_df.alias("src")
    .join(
        target_df.filter(col(
        "is_current") == lit("Y")).alias("tgt"),
        col("src.customer_id")==col("tgt.customer_id"),
        "inner"
    )
    .filter(
            col("src.salary")!=col("tgt.salary"))
    .select(col("src.*"))
)

display(changed_records_df)


# identify new records
new_records_df=(
    valid_df.alias("src")
    .join(
        target_df.filter(
            col("is_current")==lit("Y")).alias("tgt"),
        col("src.customer_id")==col("tgt.customer_id"),
        "left_anti"
        )
)
display(new_records_df) 

# expire changed records
if changed_records_df.count() > 0:
    delta_obj=DeltaTable.forName(spark,customer_config["target_table"])
    delta_obj.alias("tgt").merge(
        changed_records_df.alias("src"),
        col("src.customer_id")==col("tgt.customer_id")
    ).whenMatchedUpdate(
        set={
            "is_current":lit("N"),
            "end_date":current_timestamp()
        }
    ).execute()
else:
    print("no changed records found")

# find final df (changed records + new records)
final_df=(
    changed_records_df.union (new_records_df)
)
# add scd columns to final df
final_df=final_df.withColumn(
    "start_date",current_timestamp()
).withColumn(
    "is_current",lit("Y")
).withColumn(
    "end_date",lit(None).cast("timestamp")
)

# write into silver table
if final_df.count() > 0:
    final_df.write.format(
        "delta"
    ).mode(
        "append"
    ).option(
        "mergeSchema","true"
    ).saveAsTable(
        customer_config["target_table"]
    )
else:
    print("no new records found")

dbutils.notebook.exit("SUCCESS")

           
        
    
    
                     
 

